# Очистка и предобработка данных

Датасет: [Most Streamed Spotify Songs 2023](https://www.kaggle.com/datasets/nelgiriyewithana/top-spotify-songs-2023) (Kaggle).

В этом ноутбуке загружаем сырые данные, разбираемся с их проблемами и сохраняем чистую версию в `spotify-2023-clean.csv`. Анализ - в `notebooks/analysis.ipynb`.

## 1. Загрузка и первичный осмотр

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/spotify-2023.csv", encoding="latin-1")
df.head()

,track_name,artist(s)_name,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,streams,in_apple_playlists,...,bpm,key,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%
0,Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook",2,2023,7,14,553,147,141381703,43,...,125,B,Major,80,89,83,31,0,8,4
1,LALA,Myke Towers,1,2023,3,23,1474,48,133716286,48,...,92,C#,Major,71,61,74,7,0,10,4
2,vampire,Olivia Rodrigo,1,2023,6,30,1397,113,140003974,94,...,138,F,Major,51,32,53,17,0,31,6
3,Cruel Summer,Taylor Swift,1,2019,8,23,7858,100,800840817,116,...,170,A,Major,55,58,72,11,0,11,15
4,WHERE SHE GOES,Bad Bunny,1,2023,5,18,3133,50,303236322,84,...,144,A,Minor,65,23,80,14,63,11,6


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 953 entries, 0 to 952
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   track_name            953 non-null    str  
 1   artist(s)_name        953 non-null    str  
 2   artist_count          953 non-null    int64
 3   released_year         953 non-null    int64
 4   released_month        953 non-null    int64
 5   released_day          953 non-null    int64
 6   in_spotify_playlists  953 non-null    int64
 7   in_spotify_charts     953 non-null    int64
 8   streams               953 non-null    str  
 9   in_apple_playlists    953 non-null    int64
 10  in_apple_charts       953 non-null    int64
 11  in_deezer_playlists   953 non-null    str  
 12  in_deezer_charts      953 non-null    int64
 13  in_shazam_charts      903 non-null    str  
 14  bpm                   953 non-null    int64
 15  key                   858 non-null    str  
 16  mode               

## 2. Проблемы, которые удалось выявить после первичного осмотра

1. Колонки `streams`, `in_deezer_playlists`, `in_shazam_charts` имеют тип `str`, хотя содержат числа - в больших значениях стоят запятые-разделители тысяч (например `"2,445"`);
2. колонки `key` и `in_shazam_charts` содержат 95 и 50 пропусков соответственно;
3. в одной строке в `streams` вместо числа лежит кусок текста с описанием трека (находим её ниже);
4. названия треков с не-латинскими символами побились при чтении в `latin-1` - на числовой анализ это не влияет, поэтому оставляем как есть.

In [4]:
# ищем строки, в которых streams не приводится к числу
df[pd.to_numeric(df["streams"], errors="coerce").isna()]["streams"]

574    BPM110KeyAModeMajorDanceability53Valence75Ener...
Name: streams, dtype: str

## 3. Дубликаты

Полных дубликатов строк нет, но один и тот же трек мог попасть в датасет дважды с немного разными значениями чартов - проверяем по паре «название + исполнитель».

In [5]:
print("полных дубликатов строк:", df.duplicated().sum())

dups = df[df.duplicated(subset=["track_name", "artist(s)_name"], keep=False)]
dups[["track_name", "artist(s)_name", "released_year", "streams"]].sort_values("track_name")

полных дубликатов строк: 0


,track_name,artist(s)_name,released_year,streams
372,About Damn Time,Lizzo,2022,723894473
764,About Damn Time,Lizzo,2022,723894473
178,SNAP,Rosa Linn,2022,726307468
873,SNAP,Rosa Linn,2022,711366595
345,SPIT IN MY FACE!,ThxSoMch,2022,303216294
482,SPIT IN MY FACE!,ThxSoMch,2022,301869854
512,Take My Breath,The Weeknd,2021,130655803
616,Take My Breath,The Weeknd,2021,432702334


In [6]:
# несколько треков действительно задублировались - оставляем первое вхождение
df = df.drop_duplicates(subset=["track_name", "artist(s)_name"], keep="first").reset_index(drop=True)
len(df)

949

## 4. Приведение типов

Строку с битым `streams` восстановить нечем - числа прослушиваний в ней просто нет, поэтому удаляем её. В `in_deezer_playlists` и `in_shazam_charts` убираем запятые-разделители тысяч и переводим колонки в числа.

In [7]:
# streams: выкидываем битую строку и переводим в целые числа
df["streams"] = pd.to_numeric(df["streams"], errors="coerce")
df = df.dropna(subset=["streams"]).reset_index(drop=True)
df["streams"] = df["streams"].astype("int64")

# убираем запятые-разделители и переводим в числа
for col in ["in_deezer_playlists", "in_shazam_charts"]:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(",", ""), errors="coerce")

df.dtypes[["streams", "in_deezer_playlists", "in_shazam_charts"]]

streams                  int64
in_deezer_playlists      int64
in_shazam_charts       float64
dtype: object

## 5. Пропуски

- `key` - категориальный признак, «угадывать» тональность за трек было бы некорректно, поэтому помечаем пропуски отдельной категорией `Unknown`;
- `in_shazam_charts` - пропуск здесь по смыслу означает, что трек в чарт Shazam не попал, так что заполняем нулём.

In [8]:
df["key"] = df["key"].fillna("Unknown")
df["in_shazam_charts"] = df["in_shazam_charts"].fillna(0).astype("int64")

print("осталось пропусков во всем датафрейме:", df.isna().sum().sum())

осталось пропусков во всем датафрейме: 0


## 6. Выбросы

In [9]:
df[["streams", "bpm", "artist_count", "released_year"]].describe().round(1)

,streams,bpm,artist_count,released_year
count,9.480000e+02,948.0,948.0,948.0
mean,5.140179e+08,122.5,1.6,2018.3
std,5.679277e+08,28.0,0.9,11.0
min,2.762000e+03,65.0,1.0,1930.0
25%,1.411439e+08,99.0,1.0,2020.0
50%,2.876903e+08,120.5,1.0,2022.0
75%,6.729425e+08,140.0,2.0,2022.0
max,3.703895e+09,206.0,8.0,2023.0


Распределение `streams` сильно скошено вправо: медиана около 0.3 млрд, а максимум - 3.7 млрд. Но это не ошибки в данных, а реальные мега-хиты, и удалять их нельзя.

`bpm` лежит в реалистичных музыкальных пределах, аномалий нет. Старые треки (минимальный год релиза - 1930) тоже оставляем: это классика, которая до сих пор набирает прослушивания, и для анализа возраста трека такие наблюдения полезны.

## 7. Новые признаки

Добавляем расчетные признаки, которые пригодятся в анализе:

- `release_date` - полноценная дата релиза вместо трёх отдельных колонок;
- `track_age_years` - возраст трека на момент датасета (2023 год);
- `total_playlists` - суммарное число плейлистов по всем платформам;
- `chart_platforms` - на скольких платформах из четырёх трек попал в чарты.

In [10]:
df["release_date"] = pd.to_datetime(
    dict(year=df["released_year"], month=df["released_month"], day=df["released_day"])
)

df["track_age_years"] = 2023 - df["released_year"]

df["total_playlists"] = (
    df["in_spotify_playlists"] + df["in_apple_playlists"] + df["in_deezer_playlists"]
)

chart_cols = ["in_spotify_charts", "in_apple_charts", "in_deezer_charts", "in_shazam_charts"]
df["chart_platforms"] = (df[chart_cols] > 0).sum(axis=1)

df[["track_name", "release_date", "track_age_years", "total_playlists", "chart_platforms"]].head()

,track_name,release_date,track_age_years,total_playlists,chart_platforms
0,Seven (feat. Latto) (Explicit Ver.),2023-07-14,0,641,4
1,LALA,2023-03-23,0,1580,4
2,vampire,2023-06-30,0,1582,4
3,Cruel Summer,2019-08-23,4,8099,4
4,WHERE SHE GOES,2023-05-18,0,3304,4


## 8. Обогащение через внешний API (iTunes Search)

В исходном датасете нет жанра трека. Дотягиваем его из стороннего источника - публичного **iTunes Search API** (без ключа). По связке `track_name` + первый исполнитель ищем трек и берём поле `primaryGenreName`.

Запросы кэшируем на диск (`data/itunes_cache.json`): при повторном запуске ноутбук не ходит в сеть и воспроизводится, даже если API недоступен. Ошибки и ненайденные треки помечаем `Unknown`.

In [ ]:
import time
import json
import requests
from pathlib import Path

# кэш на диск: чтобы при перезапуске не дёргать API заново и ноутбук был воспроизводим
CACHE_PATH = Path("../data/itunes_cache.json")
cache = json.loads(CACHE_PATH.read_text(encoding="utf-8")) if CACHE_PATH.exists() else {}


def fetch_genre(track, artist):
    """Возвращает жанр трека из iTunes Search API или None, если не нашли."""
    # берём только первого исполнителя - в датасете соавторы склеены запятой
    primary_artist = artist.split(",")[0].strip()
    key = f"{track} {primary_artist}"
    if key in cache:                      # уже спрашивали - не ходим в сеть повторно
        return cache[key]
    try:
        resp = requests.get(
            "https://itunes.apple.com/search",
            params={"term": key, "entity": "song", "limit": 1},
            timeout=10,
        )
        resp.raise_for_status()
        results = resp.json().get("results", [])
        genre = results[0]["primaryGenreName"] if results else None
    except (requests.RequestException, KeyError, ValueError):
        genre = None                      # сеть упала / нет поля - помечаем как не найдено
    cache[key] = genre
    return genre


genres = []
for i, row in df.iterrows():
    genres.append(fetch_genre(row["track_name"], row["artist(s)_name"]))
    if i % 50 == 0:                       # прогресс, чтобы видеть, что не зависло
        print(f"{i}/{len(df)}")
    time.sleep(0.3)                       # вежливый троттлинг, чтобы не словить блок

df["genre"] = genres
CACHE_PATH.write_text(json.dumps(cache, ensure_ascii=False), encoding="utf-8")

matched = df["genre"].notna().sum()
print(f"
нашли жанр для {matched} из {len(df)} треков ({matched / len(df):.0%})")
df["genre"] = df["genre"].fillna("Unknown")
df["genre"].value_counts().head(10)

## 9. Сохранение результата

In [11]:
df.to_csv("../data/spotify-2023-clean.csv", index=False)
print(f"сохранили {df.shape[0]} строк и {df.shape[1]} колонок")

сохранили 948 строк и 28 колонок


## Итоги очистки

- убрали дубликаты треков и одну строку с битым значением `streams`;
- `streams`, `in_deezer_playlists`, `in_shazam_charts` приведены к числовым типам (в исходнике мешали запятые-разделители тысяч);
- пропуски обработаны: `key` -> категория `Unknown`, `in_shazam_charts` -> 0;
- экстремальные значения стримов проверили и осознанно оставили;
- добавили 4 расчетных признака: `release_date`, `track_age_years`, `total_playlists`, `chart_platforms`.

Чистый датасет лежит в `data/spotify-2023-clean.csv`